# Capstone 01: ShopSmart Starter
**ShopSmart: E-commerce Recommendation and Shopping Assistant**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
This notebook generates the project data, shows a working baseline, and seeds your golden set. Beat the baseline in your project.

* **Option A (Google Colab)** cells are active; **Option B (VS Code)** lines are commented. Toggle with `Ctrl + /`.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas rank_bm25

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas rank_bm25
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=... (git-ignored .env file)
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. Generate products and interactions

In [ ]:
import numpy as np, pandas as pd, json, os
rng = np.random.default_rng(101)
os.makedirs("data", exist_ok=True)
cats = {"kitchen": ["kettle", "rice cooker", "knife set", "tea pot"], "lighting": ["desk lamp", "floor lamp", "paper lantern"],
        "storage": ["shelf", "storage box", "shoe rack"], "textiles": ["cushion", "futon", "curtain"], "decor": ["vase", "wall clock", "bonsai pot"]}
adj = ["Aoi", "Sakura", "Kumo", "Hana", "Yama", "Mizu", "Neem", "Tulsi", "Kaveri", "Indra"]
rows = []
for i in range(2000):
    cat = rng.choice(list(cats)); kind = rng.choice(cats[cat]); name = f"{rng.choice(adj)} {kind}"
    price = int(rng.integers(500, 12000))
    rows.append({"product_id": f"P{i:04d}", "name": name, "category": cat, "price_inr": price, "price_jpy": int(price * 1.8),
                 "stock": int(rng.integers(0, 60)), "description": f"A {rng.choice(['compact', 'quiet', 'modern', 'classic', 'eco-friendly'])} {kind} for {rng.choice(['small apartments', 'family homes', 'offices', 'gifting'])}."})
products = pd.DataFrame(rows)
users = [f"U{u:04d}" for u in range(3000)]
pref = {u: rng.choice(list(cats)) for u in users}
ev = []
for _ in range(60000):
    u = rng.choice(users)
    cat = pref[u] if rng.random() < 0.7 else rng.choice(list(cats))
    p = products[products.category == cat].sample(1, random_state=int(rng.integers(1e9))).product_id.iloc[0]
    ev.append({"user_id": u, "product_id": p, "event": rng.choice(["view", "cart", "purchase"], p=[0.7, 0.2, 0.1]), "ts": int(rng.integers(0, 90))})
events = pd.DataFrame(ev)
products.to_csv("data/products.csv", index=False); events.to_csv("data/events.csv", index=False)
print(products.shape, events.shape); print(products.head(3))

## 3. Baseline: popularity vs category-aware recommendations, measured with hit rate@10
Hold out each user's last purchase; recommend from earlier activity.

In [ ]:
buys = events[events.event == "purchase"].sort_values("ts")
last = buys.groupby("user_id").tail(1)
train = events.drop(last.index)
popular = train[train.event == "purchase"].product_id.value_counts().index[:10].tolist()

cat_of = products.set_index("product_id").category
top_by_cat = {c: train[(train.event == "purchase") & train.product_id.map(cat_of).eq(c)].product_id.value_counts().index[:10].tolist() for c in products.category.unique()}
fav_cat = train.assign(cat=train.product_id.map(cat_of)).groupby("user_id").cat.agg(lambda s: s.value_counts().index[0])

def hit_rate(recommend):
    return np.mean([row.product_id in recommend(row.user_id) for row in last.itertuples()])
print(f"popularity      hit@10: {hit_rate(lambda u: popular):.3f}")
print(f"favourite cat.  hit@10: {hit_rate(lambda u: top_by_cat.get(fav_cat.get(u), popular)):.3f}")
print("Your project: beat these with co-purchase and embedding-based recommendations.")

## 4. Grounded product facts: prices come from data, not the model

In [ ]:
def product_facts(query, max_price=None, k=5):
    v = embed(products.description + " " + products.name)
    q = embed(query)[0]
    df = products.assign(score=v @ q)
    if max_price:
        df = df[df.price_inr <= max_price]
    return df.sort_values("score", ascending=False).head(k)[["product_id", "name", "price_inr", "stock"]]
print(product_facts("quiet kettle for small apartments", max_price=3000))

## 5. Golden set seed

In [ ]:
golden = [{"id": "shop-001", "query": "kettle under 3000 INR", "check": "all results are kettles priced <= 3000"},
          {"id": "shop-002", "query": "Is the first lamp in stock?", "check": "stock equals database value"},
          {"id": "shop-003", "query": "cheapest product", "check": "answer is a real product id from the catalogue"}]
with open("data/golden_seed.jsonl", "w") as f:
    for g in golden: f.write(json.dumps(g) + "\n")
print("Seeded", len(golden), "golden items. Extend to 20 or more.")

## NEW (v4): Modernisation step
Every capstone project includes one modernisation step. Below is a small legacy function from the customer's old system. Run the characterisation tests, then migrate it with OpenAI Codex in small, verified pull requests. Keep these tests green before and after each step.

In [ ]:
# Legacy code to modernise (do not edit it; pin its behaviour first)
_INDEX = []                                   # hidden global state

def legacy_search(query, products):
    """Exact-word keyword search from the old site. Case-sensitive, no ranking."""
    global _INDEX
    _INDEX = products
    words = query.split()
    return [p["id"] for p in _INDEX if all(w in p["name"] or w in p["description"] for w in words)]

sample = [{"id": "P1", "name": "Aoi kettle", "description": "A quiet kettle for small apartments."},
          {"id": "P2", "name": "Mori lamp", "description": "A modern lamp."}]

# Characterisation tests: they describe what the code DOES today, including odd behaviour.
assert legacy_search("quiet kettle", sample) == ["P1"]
assert legacy_search("Kettle", sample) == []          # case-sensitive today
assert legacy_search("", sample) == ["P1", "P2"]      # empty query returns everything today
print("Characterisation tests pass. Now migrate behind the same interface, one verified step at a time.")

---
**Ganbatte** (頑張って): *Do your best*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)